# Flush+Reset Cycle Investigation

Compares audio output from `render_params` under different flush strategies.
No dependency on the `full_flush` parameter — uses direct `plugin.process()`/`plugin.reset()` calls.

---

## Testing Matrix

| Variable | Values |
|---|---|
| **Flush strategy** | `none` (post-load only) · `post_param` · `post_render` · `all` |
| **Instance** | `isolated` (fresh plugin) · `shared` (same plugin, sequential) |
| **Params** | `saw_only` · `all_mid` · `all_max` · `random_0..9` |

| Test | Question |
|---|---|
| §3 | Do different flush strategies produce different audio? |
| §4 | Are N renders of the same config identical? |
| §5 | Does shared vs isolated plugin affect output? |
| §6 | Does render order on a shared instance change output? |

In [ ]:
import os
from pathlib import Path

import numpy as np
import rootutils
from IPython.display import HTML, Audio, display
from loguru import logger
from pedalboard import VST3Plugin

from src.data.vst.core import load_preset, make_midi_events, set_params  # noqa: E402

root = rootutils.setup_root(os.getcwd(), indicator=".project-root", pythonpath=True)
os.chdir(root)
logger.disable("src.data.vst.core")

PLUGIN_PATH = str((root / "plugins" / "Surge XT.vst3").resolve())
PRESET_PATH = "presets/surge-base.vstpreset"
SAMPLE_RATE = 44100.0
CHANNELS = 2
DURATION = 2.0
N_RENDERS = 5

print("Root:", root)
print("Plugin:", PLUGIN_PATH)
print("Exists:", Path(PLUGIN_PATH).exists())

## 2. Hardcoded test configs + helpers

In [ ]:
cfg_saw_only = {"a_osc_1_sawtooth": 0.7, "a_osc_1_volume": 0.8}

ALL_PARAMS = [
    "a_amp_eg_attack",
    "a_amp_eg_decay",
    "a_amp_eg_release",
    "a_amp_eg_sustain",
    "a_filter_1_cutoff",
    "a_filter_1_feg_mod_amount",
    "a_filter_1_resonance",
    "a_filter_2_cutoff",
    "a_filter_2_feg_mod_amount",
    "a_filter_2_resonance",
    "a_filter_eg_attack",
    "a_filter_eg_decay",
    "a_filter_eg_release",
    "a_filter_eg_sustain",
    "a_highpass",
    "a_noise_volume",
    "a_osc_1_pitch",
    "a_osc_1_sawtooth",
    "a_osc_1_width",
    "a_osc_1_sync",
    "a_osc_1_volume",
    "a_osc_1_pulse",
    "a_osc_1_triangle",
    "a_osc_2_pitch",
    "a_osc_2_sawtooth",
    "a_osc_2_width",
    "a_osc_2_sync",
    "a_osc_2_volume",
    "a_osc_2_pulse",
    "a_osc_2_triangle",
    "a_osc_3_pitch",
    "a_osc_3_sawtooth",
    "a_osc_3_width",
    "a_osc_3_sync",
    "a_osc_3_volume",
    "a_osc_3_pulse",
    "a_osc_3_triangle",
]
cfg_all_mid = {p: 0.5 for p in ALL_PARAMS}
cfg_all_max = {p: 1.0 for p in ALL_PARAMS}

cfg_random = [
    {
        "a_amp_eg_attack": 0.4924,
        "a_amp_eg_decay": 0.0193,
        "a_amp_eg_release": 0.2118,
        "a_amp_eg_sustain": 0.2232,
        "a_filter_1_cutoff": 0.7365,
        "a_filter_1_feg_mod_amount": 0.6767,
        "a_filter_1_resonance": 0.8922,
        "a_filter_2_cutoff": 0.0869,
        "a_filter_2_feg_mod_amount": 0.4219,
        "a_filter_2_resonance": 0.0298,
        "a_filter_eg_attack": 0.1684,
        "a_filter_eg_decay": 0.3891,
        "a_filter_eg_release": 0.0204,
        "a_filter_eg_sustain": 0.1988,
        "a_highpass": 0.5449,
        "a_noise_volume": 0.0,
        "a_osc_1_pitch": 0.8094,
        "a_osc_1_sawtooth": 0.0065,
        "a_osc_1_width": 0.8058,
        "a_osc_1_sync": 0.6981,
        "a_osc_1_volume": 0.3403,
        "a_osc_1_pulse": 0.1555,
        "a_osc_1_triangle": 0.9572,
        "a_osc_2_pitch": 0.5,
        "a_osc_2_sawtooth": 0.0927,
        "a_osc_2_width": 0.0967,
        "a_osc_2_sync": 0.8475,
        "a_osc_2_volume": 0.6037,
        "a_osc_2_pulse": 0.8071,
        "a_osc_2_triangle": 0.7297,
        "a_osc_3_pitch": 0.9731,
        "a_osc_3_sawtooth": 0.3785,
        "a_osc_3_width": 0.552,
        "a_osc_3_sync": 0.8294,
        "a_osc_3_volume": 0.6185,
        "a_osc_3_pulse": 0.8617,
        "a_osc_3_triangle": 0.5774,
    },
    {
        "a_amp_eg_attack": 0.0489,
        "a_amp_eg_decay": 0.2938,
        "a_amp_eg_release": 0.767,
        "a_amp_eg_sustain": 0.5291,
        "a_filter_1_cutoff": 0.9711,
        "a_filter_1_feg_mod_amount": 0.8608,
        "a_filter_1_resonance": 0.0115,
        "a_filter_2_cutoff": 0.7207,
        "a_filter_2_feg_mod_amount": 0.6817,
        "a_filter_2_resonance": 0.537,
        "a_filter_eg_attack": 0.2055,
        "a_filter_eg_decay": 0.4935,
        "a_filter_eg_release": 0.0859,
        "a_filter_eg_sustain": 0.4348,
        "a_highpass": 0.0,
        "a_noise_volume": 0.8759,
        "a_osc_1_pitch": 0.5,
        "a_osc_1_sawtooth": 0.5006,
        "a_osc_1_width": 0.1787,
        "a_osc_1_sync": 0.9126,
        "a_osc_1_volume": 0.8705,
        "a_osc_1_pulse": 0.2984,
        "a_osc_1_triangle": 0.6389,
        "a_osc_2_pitch": 0.1528,
        "a_osc_2_sawtooth": 0.7625,
        "a_osc_2_width": 0.5394,
        "a_osc_2_sync": 0.7786,
        "a_osc_2_volume": 0.5304,
        "a_osc_2_pulse": 0.0006,
        "a_osc_2_triangle": 0.3242,
        "a_osc_3_pitch": 0.5,
        "a_osc_3_sawtooth": 0.9291,
        "a_osc_3_width": 0.8787,
        "a_osc_3_sync": 0.8317,
        "a_osc_3_volume": 0.3075,
        "a_osc_3_pulse": 0.0579,
        "a_osc_3_triangle": 0.878,
    },
    {
        "a_amp_eg_attack": 0.1465,
        "a_amp_eg_decay": 0.3454,
        "a_amp_eg_release": 0.3248,
        "a_amp_eg_sustain": 0.2785,
        "a_filter_1_cutoff": 0.2498,
        "a_filter_1_feg_mod_amount": 0.9233,
        "a_filter_1_resonance": 0.4431,
        "a_filter_2_cutoff": 0.8613,
        "a_filter_2_feg_mod_amount": 0.5503,
        "a_filter_2_resonance": 0.0506,
        "a_filter_eg_attack": 0.7694,
        "a_filter_eg_decay": 0.6437,
        "a_filter_eg_release": 0.7461,
        "a_filter_eg_sustain": 0.9264,
        "a_highpass": 0.1663,
        "a_noise_volume": 0.0,
        "a_osc_1_pitch": 0.5,
        "a_osc_1_sawtooth": 0.401,
        "a_osc_1_width": 0.0586,
        "a_osc_1_sync": 0.379,
        "a_osc_1_volume": 0.9853,
        "a_osc_1_pulse": 0.2652,
        "a_osc_1_triangle": 0.7841,
        "a_osc_2_pitch": 0.5,
        "a_osc_2_sawtooth": 0.423,
        "a_osc_2_width": 0.9573,
        "a_osc_2_sync": 0.9954,
        "a_osc_2_volume": 0.5558,
        "a_osc_2_pulse": 0.7184,
        "a_osc_2_triangle": 0.1548,
        "a_osc_3_pitch": 0.5,
        "a_osc_3_sawtooth": 0.9687,
        "a_osc_3_width": 0.5792,
        "a_osc_3_sync": 0.5422,
        "a_osc_3_volume": 0.748,
        "a_osc_3_pulse": 0.0572,
        "a_osc_3_triangle": 0.5842,
    },
    {
        "a_amp_eg_attack": 0.2097,
        "a_amp_eg_decay": 0.4657,
        "a_amp_eg_release": 0.5526,
        "a_amp_eg_sustain": 0.2036,
        "a_filter_1_cutoff": 0.6342,
        "a_filter_1_feg_mod_amount": 0.264,
        "a_filter_1_resonance": 0.4885,
        "a_filter_2_cutoff": 0.9053,
        "a_filter_2_feg_mod_amount": 0.8461,
        "a_filter_2_resonance": 0.0923,
        "a_filter_eg_attack": 0.3262,
        "a_filter_eg_decay": 0.213,
        "a_filter_eg_release": 0.0027,
        "a_filter_eg_sustain": 0.7711,
        "a_highpass": 0.262,
        "a_noise_volume": 0.5517,
        "a_osc_1_pitch": 0.5,
        "a_osc_1_sawtooth": 0.0097,
        "a_osc_1_width": 0.0752,
        "a_osc_1_sync": 0.8831,
        "a_osc_1_volume": 0.9039,
        "a_osc_1_pulse": 0.5456,
        "a_osc_1_triangle": 0.8346,
        "a_osc_2_pitch": 0.1481,
        "a_osc_2_sawtooth": 0.1274,
        "a_osc_2_width": 0.3083,
        "a_osc_2_sync": 0.899,
        "a_osc_2_volume": 0.7961,
        "a_osc_2_pulse": 0.8607,
        "a_osc_2_triangle": 0.8989,
        "a_osc_3_pitch": 0.5,
        "a_osc_3_sawtooth": 0.2495,
        "a_osc_3_width": 0.1028,
        "a_osc_3_sync": 0.7801,
        "a_osc_3_volume": 0.8841,
        "a_osc_3_pulse": 0.4064,
        "a_osc_3_triangle": 0.6207,
    },
    {
        "a_amp_eg_attack": 0.5164,
        "a_amp_eg_decay": 0.2805,
        "a_amp_eg_release": 0.0539,
        "a_amp_eg_sustain": 0.6642,
        "a_filter_1_cutoff": 0.3302,
        "a_filter_1_feg_mod_amount": 0.3139,
        "a_filter_1_resonance": 0.848,
        "a_filter_2_cutoff": 0.7198,
        "a_filter_2_feg_mod_amount": 0.3003,
        "a_filter_2_resonance": 0.3093,
        "a_filter_eg_attack": 0.3145,
        "a_filter_eg_decay": 0.3098,
        "a_filter_eg_release": 0.2277,
        "a_filter_eg_sustain": 0.1273,
        "a_highpass": 0.0,
        "a_noise_volume": 0.6773,
        "a_osc_1_pitch": 0.6155,
        "a_osc_1_sawtooth": 0.3009,
        "a_osc_1_width": 0.5479,
        "a_osc_1_sync": 0.0004,
        "a_osc_1_volume": 0.2869,
        "a_osc_1_pulse": 0.4299,
        "a_osc_1_triangle": 0.58,
        "a_osc_2_pitch": 0.465,
        "a_osc_2_sawtooth": 0.4422,
        "a_osc_2_width": 0.2137,
        "a_osc_2_sync": 0.4732,
        "a_osc_2_volume": 0.9012,
        "a_osc_2_pulse": 0.796,
        "a_osc_2_triangle": 0.1697,
        "a_osc_3_pitch": 0.5,
        "a_osc_3_sawtooth": 0.5155,
        "a_osc_3_width": 0.6329,
        "a_osc_3_sync": 0.3352,
        "a_osc_3_volume": 0.8184,
        "a_osc_3_pulse": 0.7511,
        "a_osc_3_triangle": 0.6728,
    },
    {
        "a_amp_eg_attack": 0.1065,
        "a_amp_eg_decay": 0.1781,
        "a_amp_eg_release": 0.5343,
        "a_amp_eg_sustain": 0.7064,
        "a_filter_1_cutoff": 0.0642,
        "a_filter_1_feg_mod_amount": 0.4076,
        "a_filter_1_resonance": 0.5426,
        "a_filter_2_cutoff": 0.4158,
        "a_filter_2_feg_mod_amount": 0.2068,
        "a_filter_2_resonance": 0.4201,
        "a_filter_eg_attack": 0.6967,
        "a_filter_eg_decay": 0.4497,
        "a_filter_eg_release": 0.5356,
        "a_filter_eg_sustain": 0.8567,
        "a_highpass": 0.3804,
        "a_noise_volume": 0.0,
        "a_osc_1_pitch": 0.5,
        "a_osc_1_sawtooth": 0.7535,
        "a_osc_1_width": 0.8534,
        "a_osc_1_sync": 0.9534,
        "a_osc_1_volume": 0.419,
        "a_osc_1_pulse": 0.7475,
        "a_osc_1_triangle": 0.5461,
        "a_osc_2_pitch": 0.2205,
        "a_osc_2_sawtooth": 0.2194,
        "a_osc_2_width": 0.4358,
        "a_osc_2_sync": 0.029,
        "a_osc_2_volume": 0.3361,
        "a_osc_2_pulse": 0.6791,
        "a_osc_2_triangle": 0.4043,
        "a_osc_3_pitch": 0.5,
        "a_osc_3_sawtooth": 0.4674,
        "a_osc_3_width": 0.1276,
        "a_osc_3_sync": 0.6223,
        "a_osc_3_volume": 0.027,
        "a_osc_3_pulse": 0.394,
        "a_osc_3_triangle": 0.5644,
    },
    {
        "a_amp_eg_attack": 0.6026,
        "a_amp_eg_decay": 0.2673,
        "a_amp_eg_release": 0.3299,
        "a_amp_eg_sustain": 0.3706,
        "a_filter_1_cutoff": 0.506,
        "a_filter_1_feg_mod_amount": 0.3412,
        "a_filter_1_resonance": 0.8496,
        "a_filter_2_cutoff": 0.8223,
        "a_filter_2_feg_mod_amount": 0.1055,
        "a_filter_2_resonance": 0.9608,
        "a_filter_eg_attack": 0.4894,
        "a_filter_eg_decay": 0.6381,
        "a_filter_eg_release": 0.5446,
        "a_filter_eg_sustain": 0.4355,
        "a_highpass": 0.9655,
        "a_noise_volume": 0.0,
        "a_osc_1_pitch": 0.5382,
        "a_osc_1_sawtooth": 0.4835,
        "a_osc_1_width": 0.4356,
        "a_osc_1_sync": 0.731,
        "a_osc_1_volume": 0.2684,
        "a_osc_1_pulse": 0.8517,
        "a_osc_1_triangle": 0.8307,
        "a_osc_2_pitch": 0.5,
        "a_osc_2_sawtooth": 0.8816,
        "a_osc_2_width": 0.2439,
        "a_osc_2_sync": 0.4647,
        "a_osc_2_volume": 0.6103,
        "a_osc_2_pulse": 0.379,
        "a_osc_2_triangle": 0.0287,
        "a_osc_3_pitch": 0.1818,
        "a_osc_3_sawtooth": 0.2121,
        "a_osc_3_width": 0.7978,
        "a_osc_3_sync": 0.3403,
        "a_osc_3_volume": 0.8803,
        "a_osc_3_pulse": 0.7012,
        "a_osc_3_triangle": 0.2763,
    },
    {
        "a_amp_eg_attack": 0.7246,
        "a_amp_eg_decay": 0.3679,
        "a_amp_eg_release": 0.633,
        "a_amp_eg_sustain": 0.4007,
        "a_filter_1_cutoff": 0.0741,
        "a_filter_1_feg_mod_amount": 0.6294,
        "a_filter_1_resonance": 0.0536,
        "a_filter_2_cutoff": 0.1492,
        "a_filter_2_feg_mod_amount": 0.5628,
        "a_filter_2_resonance": 0.3038,
        "a_filter_eg_attack": 0.7653,
        "a_filter_eg_decay": 0.0912,
        "a_filter_eg_release": 0.5886,
        "a_filter_eg_sustain": 0.6063,
        "a_highpass": 0.2257,
        "a_noise_volume": 0.0,
        "a_osc_1_pitch": 0.5,
        "a_osc_1_sawtooth": 0.4427,
        "a_osc_1_width": 0.8602,
        "a_osc_1_sync": 0.99,
        "a_osc_1_volume": 0.3054,
        "a_osc_1_pulse": 0.621,
        "a_osc_1_triangle": 0.6096,
        "a_osc_2_pitch": 0.9476,
        "a_osc_2_sawtooth": 0.2078,
        "a_osc_2_width": 0.211,
        "a_osc_2_sync": 0.6604,
        "a_osc_2_volume": 0.1571,
        "a_osc_2_pulse": 0.1738,
        "a_osc_2_triangle": 0.0751,
        "a_osc_3_pitch": 0.5,
        "a_osc_3_sawtooth": 0.4505,
        "a_osc_3_width": 0.5938,
        "a_osc_3_sync": 0.2913,
        "a_osc_3_volume": 0.2315,
        "a_osc_3_pulse": 0.707,
        "a_osc_3_triangle": 0.703,
    },
    {
        "a_amp_eg_attack": 0.5341,
        "a_amp_eg_decay": 0.2219,
        "a_amp_eg_release": 0.7278,
        "a_amp_eg_sustain": 0.8136,
        "a_filter_1_cutoff": 0.5501,
        "a_filter_1_feg_mod_amount": 0.4548,
        "a_filter_1_resonance": 0.3145,
        "a_filter_2_cutoff": 0.3233,
        "a_filter_2_feg_mod_amount": 0.9702,
        "a_filter_2_resonance": 0.4042,
        "a_filter_eg_attack": 0.3962,
        "a_filter_eg_decay": 0.7609,
        "a_filter_eg_release": 0.5064,
        "a_filter_eg_sustain": 0.5426,
        "a_highpass": 0.0,
        "a_noise_volume": 0.0,
        "a_osc_1_pitch": 0.5,
        "a_osc_1_sawtooth": 0.7564,
        "a_osc_1_width": 0.6254,
        "a_osc_1_sync": 0.76,
        "a_osc_1_volume": 0.2036,
        "a_osc_1_pulse": 0.5492,
        "a_osc_1_triangle": 0.9277,
        "a_osc_2_pitch": 0.5,
        "a_osc_2_sawtooth": 0.6983,
        "a_osc_2_width": 0.1214,
        "a_osc_2_sync": 0.9731,
        "a_osc_2_volume": 0.6089,
        "a_osc_2_pulse": 0.2393,
        "a_osc_2_triangle": 0.1584,
        "a_osc_3_pitch": 0.5523,
        "a_osc_3_sawtooth": 0.0932,
        "a_osc_3_width": 0.9923,
        "a_osc_3_sync": 0.9129,
        "a_osc_3_volume": 0.4614,
        "a_osc_3_pulse": 0.1175,
        "a_osc_3_triangle": 0.8321,
    },
    {
        "a_amp_eg_attack": 0.3157,
        "a_amp_eg_decay": 0.5042,
        "a_amp_eg_release": 0.1189,
        "a_amp_eg_sustain": 0.4695,
        "a_filter_1_cutoff": 0.9692,
        "a_filter_1_feg_mod_amount": 0.3386,
        "a_filter_1_resonance": 0.6927,
        "a_filter_2_cutoff": 0.6498,
        "a_filter_2_feg_mod_amount": 0.8518,
        "a_filter_2_resonance": 0.8523,
        "a_filter_eg_attack": 0.6617,
        "a_filter_eg_decay": 0.2926,
        "a_filter_eg_release": 0.2438,
        "a_filter_eg_sustain": 0.7187,
        "a_highpass": 0.8724,
        "a_noise_volume": 0.0,
        "a_osc_1_pitch": 0.5,
        "a_osc_1_sawtooth": 0.6312,
        "a_osc_1_width": 0.9209,
        "a_osc_1_sync": 0.9974,
        "a_osc_1_volume": 0.7468,
        "a_osc_1_pulse": 0.434,
        "a_osc_1_triangle": 0.0984,
        "a_osc_2_pitch": 0.8726,
        "a_osc_2_sawtooth": 0.4437,
        "a_osc_2_width": 0.694,
        "a_osc_2_sync": 0.9034,
        "a_osc_2_volume": 0.046,
        "a_osc_2_pulse": 0.7961,
        "a_osc_2_triangle": 0.2934,
        "a_osc_3_pitch": 0.5,
        "a_osc_3_sawtooth": 0.1456,
        "a_osc_3_width": 0.5312,
        "a_osc_3_sync": 0.5659,
        "a_osc_3_volume": 0.7925,
        "a_osc_3_pulse": 0.17,
        "a_osc_3_triangle": 0.079,
    },
]

test_configs = [("saw_only", cfg_saw_only), ("all_mid", cfg_all_mid), ("all_max", cfg_all_max)] + [
    (f"random_{i}", cfg_random[i]) for i in range(10)
]
print(f"{len(test_configs)} test configs: {[n for n, _ in test_configs]}")

In [ ]:
import auraloss
import librosa
import torch
from loguru import logger

logger.disable("src.data.vst.core")


def fresh_plugin():
    return VST3Plugin(PLUGIN_PATH)


def normalize(audio):
    peak = np.abs(audio).max()
    return audio / peak if peak > 0 else audio


_mss_loss = auraloss.freq.MultiResolutionSTFTLoss(
    fft_sizes=[512, 1024, 2048],
    hop_sizes=[128, 256, 512],
    win_lengths=[512, 1024, 2048],
)


def mfcc_diff(a, b, sr=44100, n_mfcc=13):
    ma = librosa.feature.mfcc(y=a[0], sr=sr, n_mfcc=n_mfcc)
    mb = librosa.feature.mfcc(y=b[0], sr=sr, n_mfcc=n_mfcc)
    return np.abs(ma - mb).mean()


def spectral_loss(a, b):
    ta = torch.from_numpy(a[0:1, :]).unsqueeze(0).float()
    tb = torch.from_numpy(b[0:1, :]).unsqueeze(0).float()
    with torch.no_grad():
        return _mss_loss(ta, tb).item()


def compare(a, b, label):
    ident = np.array_equal(a, b)
    mfcc_d = mfcc_diff(a, b)
    spec_d = spectral_loss(a, b)
    print(f"  {label}: identical={ident} | mfcc={mfcc_d:.4f} | spectral={spec_d:.6f}")
    return {"ident": ident, "mfcc": mfcc_d, "spectral": spec_d}


STRATEGY_DESC = {
    "none": "post-load flush only (no post-param, no post-render)",
    "post_param": "post-load + post-param flush (no post-render)",
    "post_render": "post-load + post-render flush (no post-param)",
    "all": "post-load + post-param + post-render flush (all three)",
}


def render_direct(plugin, params, post_param_flush=False, post_render_flush=False):
    load_preset(plugin, PRESET_PATH)
    plugin.process([], 32.0, SAMPLE_RATE, CHANNELS, 2048, True)
    plugin.reset()
    set_params(plugin, params)
    if post_param_flush:
        plugin.process([], 32.0, SAMPLE_RATE, CHANNELS, 2048, True)
        plugin.reset()
    midi_events = make_midi_events(60, 100, 0.0, 1.0)
    output = plugin.process(midi_events, DURATION, SAMPLE_RATE, CHANNELS, 2048, True)
    if post_render_flush:
        plugin.process([], 32.0, SAMPLE_RATE, CHANNELS, 2048, True)
        plugin.reset()
    return output


def render_isolated(params, **kwargs):
    return render_direct(fresh_plugin(), params, **kwargs)


STRATEGIES = {
    "none": {"post_param_flush": False, "post_render_flush": False},
    "post_param": {"post_param_flush": True, "post_render_flush": False},
    "post_render": {"post_param_flush": False, "post_render_flush": True},
    "all": {"post_param_flush": True, "post_render_flush": True},
}
print("Helpers ready. Strategies:", list(STRATEGIES.keys()))

## 3. Flush strategy comparison (isolated)

**Question:** Do the optional post-param and post-render flush+reset cycles change the
rendered audio, or are they redundant?

**Why it matters:** If `none` (post-load only) produces identical audio to `all`
(post-load + post-param + post-render), we can safely skip the optional flushes and
get ~2x faster rendering. If they differ, we need to understand which flush matters
and whether the difference is perceptual or just numerical noise.

**Method:** For each of 13 param configs, render once per strategy using a fresh plugin
instance (no shared state). Compare all strategies against `all` using MFCC diff
(perceptual timbre similarity) and multi-resolution spectral loss (frequency content
similarity across time scales).

**Baseline control:** `all` is rendered **twice** per config — once at the start
(`all_start`) and once at the end (`all_end`) of each config's render batch. The
`all_start vs all_end` column shows how much `all` varies against itself due to
inherent plugin non-determinism. This is the noise floor: if `none vs all` has
similar MFCC/spectral values to `all_start vs all_end`, then the strategy difference
is within natural variation and the optional flushes are not meaningfully changing
the audio.

**How to read results:** Compare each strategy's column to the `all_start vs all_end`
column. If they are in the same range, the flush strategy does not matter. If a
strategy's values are significantly higher, that flush is affecting the audio.


In [ ]:
strategy_results = {}
for cfg_name, params in test_configs:
    strategy_results[cfg_name] = {}
    # Render "all" first as baseline
    strategy_results[cfg_name]["all_start"] = render_isolated(params, **STRATEGIES["all"])
    for strat_name, strat_kwargs in STRATEGIES.items():
        strategy_results[cfg_name][strat_name] = render_isolated(params, **strat_kwargs)
    # Render "all" again at the end
    strategy_results[cfg_name]["all_end"] = render_isolated(params, **STRATEGIES["all"])

strat_names = list(STRATEGIES.keys())
print("Strategy comparison table (mfcc | spectral vs all):")
print()
header = "{:<15}".format("Config")
for s in strat_names:
    header += f"  {s:<24}"
header += "  {:<24}".format("all_start vs all_end")
print(header)
print("-" * len(header))
for cfg_name in strategy_results:
    row = f"{cfg_name:<15}"
    for s in strat_names:
        m = mfcc_diff(strategy_results[cfg_name]["all"], strategy_results[cfg_name][s])
        sp = spectral_loss(strategy_results[cfg_name]["all"], strategy_results[cfg_name][s])
        row += "  {:<24}".format(f"{m:.2f} | {sp:.4f}")
    # all_start vs all_end
    m = mfcc_diff(strategy_results[cfg_name]["all_start"], strategy_results[cfg_name]["all_end"])
    sp = spectral_loss(
        strategy_results[cfg_name]["all_start"], strategy_results[cfg_name]["all_end"]
    )
    row += "  {:<24}".format(f"{m:.2f} | {sp:.4f}")
    print(row)

In [ ]:
for cfg_name in strategy_results:
    a_none = strategy_results[cfg_name]["none"]
    a_all = strategy_results[cfg_name]["all"]
    a_all_start = strategy_results[cfg_name]["all_start"]
    a_all_end = strategy_results[cfg_name]["all_end"]
    m = mfcc_diff(a_none, a_all)
    sp = spectral_loss(a_none, a_all)
    m_aa = mfcc_diff(a_all_start, a_all_end)
    sp_aa = spectral_loss(a_all_start, a_all_end)
    display(HTML(f"<h4>{cfg_name}</h4>"))
    display(
        HTML(
            f"<b>none vs all:</b> mfcc={m:.4f} spectral={sp:.6f} | "
            f"<b>all vs all (start/end):</b> mfcc={m_aa:.4f} spectral={sp_aa:.6f}"
        )
    )
    display(HTML("<b>none (post-load only)</b>"))
    display(Audio(a_none, rate=int(SAMPLE_RATE)))
    display(HTML("<b>all (all flushes)</b>"))
    display(Audio(a_all, rate=int(SAMPLE_RATE)))
    display(HTML("<b>all_start (rendered first)</b>"))
    display(Audio(a_all_start, rate=int(SAMPLE_RATE)))
    display(HTML("<b>all_end (rendered last)</b>"))
    display(Audio(a_all_end, rate=int(SAMPLE_RATE)))
    display(HTML("<hr>"))

## 4. Self-consistency (isolated, N renders)

**Question:** If we render the exact same config N times with fresh plugin instances,
do we get bit-identical audio every time?

**Why it matters:** Non-determinism in the renderer means identical training inputs
produce different outputs — the model sees noise rather than a clean function. If
renders are not deterministic, we need to understand the magnitude: is it floating-point
jitter (1e-7) the model will ignore, or perceptual-level variation (0.01+) that
effectively adds a random augmentation the model must learn through?

**Method:** Render  5 times per strategy, each with a fresh plugin instance.
Compare render_0 to all others (sequential pairs), and also compare even-to-even /
odd-to-odd pairs (alternate pairs) to detect binary toggle patterns.

**How to read results:** If all sequential pairs show , the strategy
is fully deterministic. If sequential pairs differ but alternate pairs (0v2, 1v3) are
closer, there is a binary internal state that flips each render. MFCC/spectral near
zero with large raw diffs means phase jitter only — perceptually identical.


In [ ]:
STRATEGY_DESC = {
    "none": "post-load flush only (no post-param, no post-render)",
    "post_param": "post-load + post-param flush (no post-render)",
    "post_render": "post-load + post-render flush (no post-param)",
    "all": "post-load + post-param + post-render flush (all three)",
}

consistency_cfg = cfg_random[0]
for strat_name, strat_kwargs in STRATEGIES.items():
    display(HTML(f"<h4>Strategy: {strat_name} (isolated x {N_RENDERS})</h4>"))
    display(HTML(f"<i>{STRATEGY_DESC[strat_name]}</i>"))
    renders = [render_isolated(consistency_cfg, **strat_kwargs) for _ in range(N_RENDERS)]
    display(HTML("<b>Sequential pairs (0 vs i):</b>"))
    for i in range(1, N_RENDERS):
        compare(renders[0], renders[i], f"render_0 vs render_{i}")
    display(HTML("<b>Alternate pairs (even/odd):</b>"))
    for i in range(0, N_RENDERS - 2):
        compare(renders[i], renders[i + 2], f"render_{i} vs render_{i + 2}")
    for i in range(N_RENDERS):
        display(HTML(f"<b>random_0 | {strat_name} | isolated | render_{i}</b>"))
        display(Audio(renders[i], rate=int(SAMPLE_RATE)))
    display(HTML("<hr>"))

## 5. Shared vs isolated instance

**Question:** Does reusing the same plugin instance across renders (as the production
pipeline does) introduce cross-render state contamination compared to using a fresh
instance each time?

**Why it matters:** The production `generate_vst_dataset.py` creates one plugin instance
and renders thousands of samples sequentially. If shared-instance renders differ from
isolated renders, prior render state is leaking into subsequent outputs — meaning sample
N's audio depends on what was rendered for samples 0..N-1, not just on sample N's params.
This would make the dataset order-dependent and non-reproducible.

**Method:** Render `random_0` 5 times on a shared instance vs 5 times isolated. Compare
shared render_0 to isolated render_0 (should be identical if no leakage). Also check
shared self-consistency (sequential and alternate pairs).

**How to read results:** If `isolated_0 vs shared_0: identical=True`, no leakage on
first render. If shared sequential pairs differ but shared alternate pairs match, the
binary toggle is present. If shared renders differ from isolated renders, the production
pipeline's output depends on execution context — a reproducibility problem.

In [ ]:
for strat_name, strat_kwargs in STRATEGIES.items():
    display(HTML(f"<h4>Strategy: {strat_name}</h4>"))
    display(HTML(f"<i>{STRATEGY_DESC[strat_name]}</i>"))

    # Render N times isolated
    renders_iso = [render_isolated(consistency_cfg, **strat_kwargs) for _ in range(N_RENDERS)]

    # Render N times shared
    p = fresh_plugin()
    renders_shared = [render_direct(p, consistency_cfg, **strat_kwargs) for _ in range(N_RENDERS)]

    for instance_label, renders in [("isolated", renders_iso), ("shared", renders_shared)]:
        display(HTML(f"<b>{instance_label} — adjacent pairs:</b>"))
        for i in range(len(renders) - 1):
            compare(renders[i], renders[i + 1], f"render_{i} vs render_{i + 1}")

        display(HTML(f"<b>{instance_label} — alternate pairs:</b>"))
        for i in range(len(renders) - 2):
            compare(renders[i], renders[i + 2], f"render_{i} vs render_{i + 2}")

        for i in range(len(renders)):
            display(HTML(f"<b>random_0 | {strat_name} | {instance_label} | render_{i}</b>"))
            display(Audio(renders[i], rate=int(SAMPLE_RATE)))

    # Cross: isolated vs shared per render index
    display(HTML("<b>Cross: isolated_i vs shared_i</b>"))
    for i in range(N_RENDERS):
        compare(renders_iso[i], renders_shared[i], f"isolated_{i} vs shared_{i}")
    display(HTML("<hr>"))

## 6. Ordering dependence

**Question:** If we render configs A, B, C, D on a shared instance, does the order
(A→B→C→D vs D→C→B→A) change the output for any individual config?

**Why it matters:** The production pipeline renders shards in a fixed order determined
by the random seed. If ordering affects output, then shuffling the dataset or splitting
into different shard sizes would produce different audio for the same params — making
the dataset non-reproducible and the training signal noisy. This test also reveals
whether the post-render flush is needed: if ordering matters without it but not with it,
the post-render flush serves a real purpose.

**Method:** Render 4 configs (saw, mid, max, random_0) in 3 different orders on shared
instances, plus isolated ground truth. Compare each config's output across orders and
against isolated. Test both `none` and `all` strategies.

**How to read results:** If `order_A vs order_B: identical=True` for all configs, ordering
doesn't matter and the post-render flush is redundant. If they differ under `none` but
match under `all`, the post-render flush prevents cross-config leakage and is needed for
reproducibility. Example: if `saw_only | order_A vs order_B: norm max=0.2` under `none`
but `norm max=0.0` under `all`, prior config state leaks without the post-render flush.

In [ ]:
order_configs = [
    ("saw_only", cfg_saw_only),
    ("all_mid", cfg_all_mid),
    ("all_max", cfg_all_max),
    ("random_0", cfg_random[0]),
]
orders = {"A": [0, 1, 2, 3], "B": [3, 2, 1, 0], "C": [2, 0, 3, 1]}

for strat_name, strat_kwargs in [("none", STRATEGIES["none"]), ("all", STRATEGIES["all"])]:
    display(HTML(f"<h3>Strategy: {strat_name}</h3>"))
    order_renders = {}
    for order_name, order in orders.items():
        p = fresh_plugin()
        order_renders[order_name] = {}
        for idx in order:
            cfg_name, params = order_configs[idx]
            order_renders[order_name][cfg_name] = render_direct(p, params, **strat_kwargs)
    iso = {name: render_isolated(params, **strat_kwargs) for name, params in order_configs}
    for cfg_name, _ in order_configs:
        display(HTML(f"<h4>{cfg_name}</h4>"))
        for oname in orders:
            compare(iso[cfg_name], order_renders[oname][cfg_name], f"isolated vs order_{oname}")
        for o1, o2 in [("A", "B"), ("A", "C"), ("B", "C")]:
            compare(
                order_renders[o1][cfg_name],
                order_renders[o2][cfg_name],
                f"order_{o1} vs order_{o2}",
            )
        display(HTML(f"<b>{cfg_name} | isolated</b>"))
        display(Audio(iso[cfg_name], rate=int(SAMPLE_RATE)))
        for oname in orders:
            display(HTML(f"<b>{cfg_name} | order_{oname}</b>"))
            display(Audio(order_renders[oname][cfg_name], rate=int(SAMPLE_RATE)))
        display(HTML("<hr>"))

## 7. Summary heatmap

Normalized max absolute difference across all configs and strategy pairs.
Dark = identical, bright = divergent.

In [ ]:
import matplotlib.pyplot as plt

strat_names = list(STRATEGIES.keys())
compare_against = [s for s in strat_names if s != "all"] + ["all_start vs all_end"]
cfg_names = list(strategy_results.keys())

mat_mfcc = np.zeros((len(cfg_names), len(compare_against)))
mat_spec = np.zeros((len(cfg_names), len(compare_against)))

for r, cfg_name in enumerate(cfg_names):
    for c, strat in enumerate(compare_against):
        if strat == "all_start vs all_end":
            a = strategy_results[cfg_name]["all_start"]
            b = strategy_results[cfg_name]["all_end"]
        else:
            a = strategy_results[cfg_name]["all"]
            b = strategy_results[cfg_name][strat]
        mat_mfcc[r, c] = mfcc_diff(a, b)
        mat_spec[r, c] = spectral_loss(a, b)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, matrix, title in [
    (axes[0], mat_mfcc, "MFCC diff vs all"),
    (axes[1], mat_spec, "Spectral loss vs all"),
]:
    im = ax.imshow(matrix, aspect="auto", cmap="Reds")
    ax.set_xticks(range(len(compare_against)))
    ax.set_xticklabels(compare_against, fontsize=9, rotation=30, ha="right")
    ax.set_yticks(range(len(cfg_names)))
    ax.set_yticklabels(cfg_names, fontsize=9)
    ax.set_xlabel("Strategy")
    ax.set_ylabel("Config")
    ax.set_title(title)
    for r in range(len(cfg_names)):
        for c in range(len(compare_against)):
            v = matrix[r, c]
            color = "white" if v > matrix.max() * 0.6 else "black"
            ax.text(c, r, f"{v:.3f}", ha="center", va="center", fontsize=7, color=color)
    plt.colorbar(im, ax=ax)
plt.suptitle("Each strategy compared to all (isolated renders) + all vs all baseline", fontsize=12)
plt.tight_layout()
plt.show()

## 8. Consistency heatmaps

For each strategy, render random_0 N times isolated and shared.
Three heatmaps:
- **Isolated vs shared**: render_i isolated vs render_i shared
- **Shared sequential**: render_i vs render_{i+1} (adjacent pairs)
- **Shared alternate**: render_i vs render_{i+2} (skip-one pairs, detects binary toggle)


In [ ]:
N = N_RENDERS
consistency_renders = {}
for strat_name, strat_kwargs in STRATEGIES.items():
    iso = [render_isolated(consistency_cfg, **strat_kwargs) for _ in range(N)]
    p = fresh_plugin()
    shared = [render_direct(p, consistency_cfg, **strat_kwargs) for _ in range(N)]
    consistency_renders[strat_name] = {"iso": iso, "shared": shared}

strat_list = list(STRATEGIES.keys())


def build_matrix(renders_key, pair_fn, n_pairs):
    result = {m: np.zeros((n_pairs, len(strat_list))) for m in ["mfcc", "spectral"]}
    for c, strat in enumerate(strat_list):
        for r in range(n_pairs):
            i, j = pair_fn(r)
            a = consistency_renders[strat][renders_key][i]
            b = consistency_renders[strat][renders_key][j]
            result["mfcc"][r, c] = mfcc_diff(a, b)
            result["spectral"][r, c] = spectral_loss(a, b)
    return result


def build_cross_matrix():
    result = {m: np.zeros((N, len(strat_list))) for m in ["mfcc", "spectral"]}
    for c, strat in enumerate(strat_list):
        for r in range(N):
            a = consistency_renders[strat]["iso"][r]
            b = consistency_renders[strat]["shared"][r]
            result["mfcc"][r, c] = mfcc_diff(a, b)
            result["spectral"][r, c] = spectral_loss(a, b)
    return result


def plot_metrics(metrics_dict, row_labels, col_labels, suptitle):
    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(row_labels) * 0.7 + 2)))
    titles = ["MFCC diff", "Spectral loss"]
    for ax, (metric, matrix), title in zip(axes, metrics_dict.items(), titles):
        im = ax.imshow(matrix, aspect="auto", cmap="Reds")
        ax.set_xticks(range(len(col_labels)))
        ax.set_xticklabels(col_labels, fontsize=10)
        ax.set_yticks(range(len(row_labels)))
        ax.set_yticklabels(row_labels, fontsize=9)
        ax.set_title(title, fontsize=10)
        for r in range(matrix.shape[0]):
            for c in range(matrix.shape[1]):
                v = matrix[r, c]
                color = "white" if v > matrix.max() * 0.6 else "black"
                ax.text(c, r, f"{v:.3f}", ha="center", va="center", fontsize=8, color=color)
        plt.colorbar(im, ax=ax)
    plt.suptitle(suptitle, fontsize=12)
    plt.tight_layout()
    plt.show()


# Isolated vs Shared
cross = build_cross_matrix()
plot_metrics(
    cross,
    [f"render_{i}" for i in range(N)],
    strat_list,
    "Isolated vs Shared: render_i compared across instance types",
)

# Shared adjacent
adj = build_matrix("shared", lambda r: (r, r + 1), N - 1)
plot_metrics(
    adj,
    [f"r{i} vs r{i + 1}" for i in range(N - 1)],
    strat_list,
    "Shared: adjacent renders (per-render drift)",
)

# Shared alternate
alt = build_matrix("shared", lambda r: (r, r + 2), N - 2)
plot_metrics(
    alt,
    [f"r{i} vs r{i + 2}" for i in range(N - 2)],
    strat_list,
    "Shared: alternate renders (binary toggle detection)",
)

# Isolated adjacent (baseline)
iso_adj = build_matrix("iso", lambda r: (r, r + 1), N - 1)
plot_metrics(
    iso_adj,
    [f"r{i} vs r{i + 1}" for i in range(N - 1)],
    strat_list,
    "Isolated: adjacent renders (baseline non-determinism)",
)